In [ ]:
!pip install nbstripout
!nbstripout Chatbot_Evaluation\Chatbot_Evaluation.ipynb

In [ ]:
import os
os.getcwd()

In [ ]:
!pip install --pre -U langchain langchain-openai langchain_community langchain_core langchain_text_splitters  unstructured langchain_huggingface langchain_cohere

## the model
we will use local model as a judge
we will use "Qwen/Qwen2.5-7B-Instruct"


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LlamaTokenizer ,LlamaForCausalLM ,GenerationConfig ,pipeline ,AutoTokenizer ,AutoModelForCausalLM
import torch
from langchain_huggingface  import HuggingFacePipeline,ChatHuggingFace

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Chatbot Evaluation

In [ ]:
from google.colab import userdata
import os
langsmith_api_key = userdata.get('LANGSMITH')

os.environ['LANGCHAIN_TRACING'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = langsmith_api_key


### Define dataset

In [ ]:
from langsmith import Client

client = Client()

# define :database : the test case

database_name = "Chatbot evaluation"

dataset = client.create_dataset(
        dataset_name=database_name,
        description="Chatbot evaluation"
                                )

In [ ]:
dataset.id

In [ ]:
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs":{"question":"what is langchain"},
            "outputs":{"answer":"A framework for building LLM applications"}
        },
        {
            "inputs":{"question":"What is LangSmith?"},
            "outputs":{"answer":"A platform for observing and evaluating LLM applications"}
        },
        {
            "inputs":{"question":"What is OpenAI?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs":{"question":"What is Google?"},
            "outputs":{"answer":"A technology company known for search"}
        },

        {
            "inputs":{"question":"What is Mistral?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs": {"question": "What is Hugging Face?"},
            "outputs": {"answer": "A company that provides tools and models for natural language processing"}
        },
        {
            "inputs": {"question": "What is TensorFlow?"},
            "outputs": {"answer": "An open-source machine learning framework developed by Google"}
        },
        {
            "inputs": {"question": "What is PyTorch?"},
            "outputs": {"answer": "An open-source machine learning library used for deep learning applications"}
        },
        {
            "inputs": {"question": "What is Anthropic?"},
            "outputs": {"answer": "A company that develops AI systems and large language models"}
        },
        {
            "inputs": {"question": "What is Cohere?"},
            "outputs": {"answer": "A company that provides language AI models and APIs for developers"}
        }

    ]
)

### Define Metrics (LLM As A Judge)
Qwen/Qwen2.5-7B-Instruct

In [ ]:

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading a student's answer.
          Question: {inputs['question']}
          Correct Answer: {reference_outputs['answer']}
          Student Answer: {outputs['response']}

          Instructions:
          - Respond with ONLY one word.
          - Allowed answers: CORRECT or INCORRECT
          - Do not explain.

          grade:"""

      messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]

      text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
      )

      inputs = tokenizer([text], return_tensors="pt").to(model.device)

      outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1
        )

      generated_ids = outputs[0][inputs.input_ids.shape[1]:]

      response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
      )



      return response.strip() == "CORRECT"

In [ ]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs:dict, reference_outputs:dict) -> bool:
    response = outputs.get('response', "")
    return int(len(response) < 2 * len(reference_outputs['answer']))

### Run Evaluations

In [ ]:
defult_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str ,instructions:str = defult_instructions) -> str:
  messages=[
           {"role": "system", "content": instructions},
           {"role": "user", "content": question}
       ]

  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
  )

  inputs = tokenizer([text], return_tensors="pt").to(model.device)

  outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1
    )

  generated_ids = outputs[0][inputs.input_ids.shape[1]:]

  response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
  )

  return response.strip()

In [ ]:
### Call my_app for every datapoints

def ls_target(inputs):
    try:
        result = my_app(inputs['question'])
        return {"response": result}
    except Exception as e:
        return {"response": "", "error": str(e)}


In [ ]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target, ## Your AI system
    data = database_name,
    evaluators = [correctness,concision],
    experiment_prefix="Qwen/Qwen2.5-7B-Instruct"

)